# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset on clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) accessible at:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the mlcroissant library (if needed)
!pip install mlcroissant

## 1. Data Loading
Load the Croissant schema and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', '<unknown>')}")
print(f"Description: {getattr(metadata, 'description', '<unknown>')}")
print(f"Identifier: {getattr(metadata, 'identifier', '<unknown>')}")
print(f"Number of record sets: {len(getattr(metadata, 'record_set', []))}")

## 2. Data Overview
List the available record sets and their fields using their Croissant `@id` values for reference.

In [ ]:
# List available record sets and fields (by @id)
record_sets = getattr(metadata, 'record_set', [])

if not record_sets:
    # Try alternate attribute if empty (older Croissant: 'recordSets')
    record_sets = getattr(metadata, 'recordSets', [])

print(f"Found {len(record_sets)} record sets.\n")

for rs in record_sets:
    rs_id = getattr(rs, '@id', getattr(rs, 'id', '<unknown>'))
    rs_name = getattr(rs, 'name', '<unknown name>')
    print(f"Record set @id: {rs_id}")
    print(f"  Name: {rs_name}")
    field_objs = getattr(rs, 'field', [])
    if not isinstance(field_objs, list):
        field_objs = [field_objs]
    print("  Fields:")
    for f in field_objs:
        f_id = getattr(f, '@id', getattr(f, 'id', '<unknown>'))
        f_name = getattr(f, 'name', '<unknown name>')
        print(f"    - @id: {f_id:40} | Name: {f_name}")
    print()

## 3. Data Extraction
Load tabular data from record set(s) using their `@id`. Extract data into pandas DataFrames for further analysis.

In [ ]:
# Collect record set @ids for extraction
rs_ids = [getattr(rs, '@id', getattr(rs, 'id', None)) for rs in record_sets]
# Remove None values if any
rs_ids = [x for x in rs_ids if x is not None]

dataframes = {}

for rs_id in rs_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded record set @id: {rs_id} (n={len(records)})")
        else:
            print(f"WARNING: No records found for record set @id: {rs_id}")
    except Exception as e:
        print(f"ERROR loading record set @id {rs_id}: {e}")

# Display first DataFrame details if available
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set @id {first_rs_id}:\n", dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group data using one of the numeric fields in the loaded dataset. All field references use their Croissant `@id` values.

In [ ]:
# Example: Use the first record set for EDA
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id].copy()
    print(f"Performing EDA on record set @id: {rs_id}")

    # Identify numeric field(s) by @id (using overview from section 2)
    # Here, we pick the first float/int-looking column
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field (by @id): {numeric_field}")
    else:
        print("No numeric fields present.")
        numeric_field = None

    # Filter for records with value > threshold
    if numeric_field:
        threshold = df[numeric_field].median()  # use median as a threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered on {numeric_field} > {threshold}, remaining rows: {len(filtered_df)}")

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Pick a group field (first string/categorical field)
        group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field (by @id): {group_field}")
            # Group and aggregate mean
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print("Grouped Data:")
            print(grouped_df.head())
        else:
            print("No group (categorical) field found.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group field if available
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset on second primary colorectal cancer survivors using the `mlcroissant` library:
- We loaded the dataset schema and records programmatically in accordance with the Croissant specification.
- All tabular data and fields were referenced using their Croissant `@id`.
- We demonstrated basic exploratory analysis and visualization for typical data processing workflows.

**Key findings and visualization outputs depend on the field structure and actual numeric and categorical variables available for the given record sets.**

For more advanced analysis, refer to the detailed data dictionary and explore other record sets and fields by their `@id` values as needed.